# Blocked-query patterns: the dnsblock exploration

> **Finding:** on a Pi-hole/dnsmasq log, client attribution of blocked queries is a
> join, not a fact: disposition lines carry no client, so a query is only ever
> *associated* with a name's recorded block outcomes. A name's blocked-ness is a
> ratio rather than a boolean (list churn, CNAME paths, and per-group policy all mix
> dispositions), and an honest novelty noun must separate a name with *no prior
> logged handling* from one the blocklist merely caught up with while it was already
> being handled.

This notebook works those mechanics out on the deterministic demo corpus so the method
is reproducible by anyone. It is a mechanics-and-hypothesis notebook for the planned
`dnsblock` detector (see `docs/ROADMAP.md`, "Known-bad access patterns"): it
demonstrates data mechanics only; no detection power, no calibration, no baseline
claims.

## Data disclosure

This notebook is **demo-only**: it reads the seeded demo corpus (`demo/gen_corpus.py`)
and nothing else. It contains no real log records, client addresses, hostnames, or
operational domains; the demo's names live in reserved documentation space
(`example.com`-class, RFC 5737 addresses, and a seeded random label under a real TLD
as the DGA stand-in). Printed tables mask source addresses behind stable opaque labels
even on demo data, so the cells stay aggregate/masked by construction. Committed
outputs are stripped. Exploring a real archive belongs to the detector's own
measurement instruments, which carry windowing, bounds, and masking this notebook does
not.

## Question and scope

The planned `dnsblock` detector asks: **who queries the domains your own Pi-hole already
blocks, with what persistence, across how many clients?** It uses the operator's
blocklist verdicts, never a feed sigwood ships. This notebook establishes the *data
mechanics* that question sits on:

1. what the dnsmasq event taxonomy actually carries (and that `src` exists only on
   `query` lines),
2. how blocked-ness must be derived (a per-name disposition aggregate, not a flag),
3. why novelty needs the prior-handling / same-day-ambiguous / no-prior-handling
   name classes,
4. how persistence, spread, and retry-cadence dispersion are measured.

**Status:** hypothesis-generation only, demo-only. Nothing here freezes a detector
gate; calibration numbers for the shipped detector come from its own preregistered
product-path measurement runs over real archives, recorded with the design.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from sigwood.common import loader
from sigwood.common.tld import roll_domain  # shared OFFLINE suffix owner - never
                                            # raw tldextract (that path may try the
                                            # network or a user cache)

# Demo-only by contract (see the data disclosure above): the corpus is generated by
#   python demo/gen_corpus.py        # materializes demo/corpus (gitignored)
# Resolve robustly whether Jupyter started at the repo root or in notebooks/.
_CANDIDATES = [Path("demo/corpus/pihole"), Path("../demo/corpus/pihole")]
LOG_DIR = next((p for p in _CANDIDATES if p.is_dir()), _CANDIDATES[0])

BLOCK_TYPES = {"gravity_blocked", "regex_blocked"}


def mask_addresses(index_or_series):
    # stable opaque labels for source addresses, first-seen order
    labels: dict = {}

    def lab(a):
        if a not in labels:
            labels[a] = f"client-{len(labels):02d}"
        return labels[a]

    return [lab(a) for a in index_or_series]

## Load through the product loader

`load_pihole` discovers `pihole*.log*` under the directory, parses each line with the
product dnsmasq grammar, and returns the canonical six-column frame
(`ts, src, query, event_type, qtype, host`). Everything downstream uses exactly what a
shipped detector would see, with no bespoke parsing.

In [ ]:
df = loader.load_pihole(LOG_DIR, show_progress=False)
# An empty load must stop the exploration here - every later cell would print
# an all-zero "success" otherwise.
assert len(df), (
    f"no rows loaded from {LOG_DIR.resolve()} - generate the demo corpus first "
    "(python demo/gen_corpus.py)"
)
print(f"{len(df):,} rows, {df['ts'].nunique():,} distinct timestamps")
print("\nevent taxonomy census:")
for et, n in df["event_type"].value_counts().items():
    print(f"  {et:18s} {n:>8,}")

## The blocked-name ledger

`gravity_blocked` and `regex_blocked` are two mechanisms of one outcome: the Pi-hole
refused to resolve. The parser keeps them distinct (a real config detail); the consumer
collapses them into one *blocked* notion. Registrable-domain grouping (`tldextract`)
matters: blocklists catch many names under one family, and family grain is where
behavioral claims stay legible.

In [ ]:
blocked_events = df[df["event_type"].isin(BLOCK_TYPES)]
blocked_names = {
    q.lower().rstrip(".") for q in blocked_events["query"].dropna()
    if isinstance(q, str)
}
fam = {n: roll_domain(n, "domain") for n in blocked_names}
print(f"block events: {len(blocked_events):,}")
print(f"distinct blocked names: {len(blocked_names):,}")
print(f"registrable families:   {len(set(fam.values())):,}")
print("\nby mechanism:")
print(blocked_events["event_type"].value_counts().to_string())

## Client attribution is a join, and it is approximate

dnsmasq populates the client (`src`) only on `query` lines; the disposition lines
(`gravity blocked ... is 0.0.0.0`) carry the domain but no client, and the default log
format has no query serial numbers to correlate on. So "who queried a blocked name" is a
**domain-level join**: `(client, name)` pairs come from query events, blocked-ness from
the name's disposition aggregate.

The join's error envelope is *mixed disposition*: one name can be blocked part of the
window and forwarded the rest (blocklist refreshes mid-window), blocked for one client
group and allowed for another (Pi-hole group management is per-client policy), or
answered through CNAME-inspection paths. On real archives a noticeable minority of
blocked names show both dispositions in one window, so blocked-ness must be carried as a
ratio or per-day record, never a boolean.

In [ ]:
q = df[df["event_type"] == "query"].dropna(subset=["query", "src"]).copy()
q["name"] = q["query"].map(
    lambda s: s.lower().rstrip(".") if isinstance(s, str) else None)
qb = q[q["name"].isin(blocked_names)]
# "attributed": queries for names for which Pi-hole logged a blocked outcome
# SOMEWHERE in this data - the LOOSE whole-corpus association, demonstrated for
# mechanics only. The shipped detector uses stricter, window-bounded association
# rules; and no join can prove any INDIVIDUAL query was answered with a block,
# which is why the noun is never "blocked queries".
print(f"query events: {len(q):,}; attributed query count (loose): {len(qb):,}")

pairs = qb.groupby(["src", "name"]).size().sort_values(ascending=False)
print(f"(address, blocked name) pairs: {len(pairs):,}")

# disposition per blocked name: sum raw event counts FIRST, then divide - never
# average per-name ratios.
disp = df.copy()
disp["name"] = disp["query"].map(
    lambda s: s.lower().rstrip(".") if isinstance(s, str) else None)
disp = disp[disp["name"].isin(blocked_names)]
mix = disp.groupby("name")["event_type"].value_counts().unstack(fill_value=0)
blk = mix.reindex(columns=sorted(BLOCK_TYPES), fill_value=0).sum(axis=1)
res = mix.get("forwarded", 0) + mix.get("cached", 0)
ratio = (blk / (blk + res).clip(lower=1)).rename("blocked_share")
mixed = ratio[(blk > 0) & (res > 0)]
print(f"names with BOTH block and forwarded/cached events: {len(mixed)}")

# per-address attributed pressure, self-relative; addresses masked (the
# aggregate/masked-prints contract from the data disclosure)
per_client = q.groupby("src").size().rename("queries").to_frame()
per_client["attributed_q"] = qb.groupby("src").size()
per_client = per_client.fillna(0).astype(int)
per_client["own_share_pct"] = (
    100 * per_client["attributed_q"] / per_client["queries"].clip(lower=1)
).round(2)
per_client = per_client.sort_values("attributed_q", ascending=False)
per_client.index = mask_addresses(per_client.index)
print("\nper-address attributed pressure (top 8, masked):")
print(per_client.head(8).to_string())

## Persistence and spread: the mechanics

The hypothesis this cell exists to test on real data (not established here): the
persistence distribution of `(address, blocked name)` pairs may be strongly bimodal:
a transient head and an always-on core of retrying software. If that holds, raw
persistence is a baseline property rather than an anomaly, and a detector must key on
*change* against it. This notebook only demonstrates the measuring mechanics; day
bucketing and spread counting; the demo corpus covers a single day, so day-grain
persistence is degenerate here by construction.

In [ ]:
# UTC analysis days: day-grained grouping keys are derived in UTC (render
# timezone is a separate, display-side concern).
qb2 = qb.copy()
qb2["utc_day"] = pd.to_datetime(qb2["ts"], unit="s", utc=True).dt.strftime("%Y%m%d")
qb2["family"] = qb2["query"].map(
    lambda q: roll_domain(q.lower().rstrip("."), "domain"))
pair_days = qb2.groupby(["src", "family"])["utc_day"].nunique()
print("(address, family) persistence (distinct active UTC days -> pairs):")
print(pair_days.value_counts().sort_index().to_string())

spread = qb2.groupby("family")["src"].nunique()
print("\nfamily spread (distinct client addresses -> families):")
print(spread.value_counts().sort_index().to_string())

## The honest novelty noun: prior handling vs no prior handling

"An address started reaching for a name with no previously logged handling" and "the
blocklist started covering a name Pi-hole was already handling" are opposite stories,
the first may be a behavior change at the client, the second is a list change at the
server (gravity refreshes weekly by default, and the text log records no list updates).
A novelty claim that conflates them manufactures findings out of routine list
maintenance.

The split is computable from the log at name grain: a name whose first blocked day is
preceded by a forwarded/cached **handling** day (handling, not "resolution": a cached
answer can carry a negative result) is a `prior_handling` name; handling seen only on
the first block day itself is `same_day_handling_ambiguous` (the log cannot prove
same-day ordering); neither is `no_prior_handling`. Only the last class can support a
behavior-flavored claim, and any "first" is relative to the loaded data, a fact a
detector must say out loud.

In [ ]:
# Vectorized, UTC-analysis-day grain, strict earlier-day precedence. Grouped
# aggregations keep this O(rows) - never a per-name scan of the whole frame.
ev = df.dropna(subset=["ts"]).copy()
ev["name"] = ev["query"].map(
    lambda q: q.lower().rstrip(".") if isinstance(q, str) else None)
ev = ev[ev["name"].isin(blocked_names)]
ev["utc_day"] = pd.to_datetime(ev["ts"], unit="s", utc=True).dt.strftime("%Y%m%d")

first_block_day = (
    ev[ev["event_type"].isin(BLOCK_TYPES)].groupby("name")["utc_day"].min()
)
handling_days = (
    ev[ev["event_type"].isin({"forwarded", "cached"})]
    .groupby("name")["utc_day"].agg(["min", lambda s: set(s)])
    .rename(columns={"<lambda_0>": "days"})
)
cls = {}
for name, fbd in first_block_day.items():
    if name in handling_days.index:
        hmin = handling_days.loc[name, "min"]
        hdays = handling_days.loc[name, "days"]
        cls[name] = ("prior_handling" if hmin < fbd
                     else "same_day_handling_ambiguous" if fbd in hdays
                     else "no_prior_handling")
    else:
        cls[name] = "no_prior_handling"
print(pd.Series(cls).value_counts().to_string())
print("(a one-day demo corpus makes nearly everything no_prior_handling by "
      "construction; the classes exist so a multi-week archive can keep list "
      "churn and unprovable same-day ordering out of a novelty claim)")

## Retry cadence: the instrument, not a gate

The folk model says blocked malware retries on a metronome while human-driven traffic
does not. Whether that separates anything on a given network is an empirical question
this notebook does not answer; the cell below only builds the instrument: the
inter-arrival coefficient of variation per pair. Treat any threshold on it as a
hypothesis to be swept and adjudicated on real data through the detector's own
measurement path, never as a verdict.

In [ ]:
MIN_GAPS = 20
rows = []
for (src, name), grp in qb.dropna(subset=["ts"]).groupby(["src", "name"]):
    ts = grp["ts"].sort_values().to_numpy()
    if len(ts) < MIN_GAPS + 1:
        continue
    gaps = pd.Series(ts).diff().dropna()
    gaps = gaps[gaps < 6 * 3600]  # within-session gaps only
    if len(gaps) >= MIN_GAPS and gaps.mean() > 0:
        rows.append((src, name, len(gaps),
                     round(gaps.mean(), 1),
                     round(gaps.std(ddof=0) / gaps.mean(), 2)))
cad = pd.DataFrame(rows, columns=["src", "name", "gaps", "mean_gap_s", "cv"])
if len(cad):
    cad = cad.sort_values("cv")
    cad["src"] = mask_addresses(cad["src"])          # masked-prints contract
    cad["family"] = cad["name"].map(
        lambda n: roll_domain(n, "domain"))
    print(cad.drop(columns=["name"]).head(10).to_string(index=False))
    print(f"\npairs measured: {len(cad)}; CV quantiles: "
          f"{cad['cv'].quantile([0.1, 0.5, 0.9]).round(2).to_dict()}")
else:
    print("no pair reaches the gap floor on this corpus - expected on the 24h demo "
          "(the distribution question belongs to the detector's measurement path)")

## Interpretation: what this establishes, and what it does not

Established here, mechanically, on the demo corpus:

- the attribution model (queries are *associated* with a name's recorded block
  outcomes through a join; disposition is carried as a summed-then-divided ratio),
- the prior-handling / same-day-ambiguous / no-prior-handling name classes as the
  honest novelty vocabulary,
- the measuring mechanics for persistence, spread, and retry-cadence dispersion.

**Not established here:** detection power, thresholds, severity semantics, any benign
or malicious baseline, or any claim about real traffic; this corpus is one synthetic
day. Seeded synthetic data can only ever establish trip direction (that a mechanism
fires on the shape it was built for), never real-world rates. The `dnsblock`
detector's frozen contracts and calibration numbers come from its own preregistered,
product-path measurement runs over real archives, not from re-running this notebook.